In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('./SQL_GRIT_L1.csv')

df.head()

,Question Creation Date,Question Last Update Date,Question ID,Question Topic,Question Subtopic,Question Content,Short Text,Options Data,Question MetaData,Options MetaData,Question Pool,Question Difficulty,Source of Question,Question Offline Status,Question Company Source,Course Tag of Question,Module Tag of Question,Unit Tag of Question,Job ID,Extra Tags
0,18 Aug 2025,26 May 2026,fff150a8e3f2419eb1b311d1724a3f99,TOPIC_SQL_MCQ,SUB_TOPIC_SQL_DDL,Which SQL statement correctly creates a `train...,NaN,"[{""option_content"":""```sql\nCREATE TABLE train...",NaN,NaN,POOL_1,DIFFICULTY_MEDIUM,SOURCE_GPT,IN_OFFLINE_EXAM,NaN,COURSE_Introduction_to_Databases,MODULE_Modelling_Databases,UNIT_Modelling_Database_2_FINAL,NaN,"CATEGORY_CODE_ANALYSIS, IS_PUBLIC, NIAT, TESTI..."
1,18 Aug 2025,26 May 2026,fff150a8e3f2419eb1b311d1724a3f99,TOPIC_SQL_MCQ,SUB_TOPIC_SQL_DDL,Which SQL statement correctly creates a `train...,NaN,"[{""option_content"":""```sql\nCREATE TABLE train...",NaN,NaN,POOL_1,DIFFICULTY_MEDIUM,SOURCE_GPT,IN_OFFLINE_EXAM,NaN,COURSE_Introduction_to_Databases,MODULE_Modelling_Databases,UNIT_Modelling_Database_2,NaN,"CATEGORY_CODE_ANALYSIS, IS_PUBLIC, NIAT, TESTI..."
2,18 Aug 2025,26 May 2026,ffc207e0bf404b2887f055dd6bdac117,TOPIC_SQL_MCQ,SUB_TOPIC_SQL_FUNCTIONS,Table: project_budgets\r\n\r\n| projectId | pr...,NaN,"[{""option_content"":""| total_budget |\r\n|-----...",NaN,",",POOL_1,DIFFICULTY_EASY,SOURCE_GPT,IN_OFFLINE_EXAM,NaN,COURSE_Introduction_to_Databases,MODULE_SQL_Expressions_and_Functions,UNIT_SQL_Expression,NaN,"CATEGORY_CODE_ANALYSIS, IS_PUBLIC, NIAT_CUMULA..."
3,24 Jul 2025,20 Apr 2026,ff9c7881e55346ab80c436ecc451005b,TOPIC_SQL_MCQ,SUB_TOPIC_SQL_DATA_TYPES,What is the correct SQL data type to store a d...,NaN,"[{""option_content"":""DATE"",""is_correct_option"":...",NaN,NaN,POOL_1,DIFFICULTY_EASY,SOURCE_GPT,IN_OFFLINE_EXAM,NaN,COURSE_Introduction_to_Databases,MODULE_Introduction_To_SQL,UNIT_Create_Table,NaN,"CATEGORY_CODE_ANALYSIS, IN_LP, IS_PUBLIC, NIAT..."
4,22 Jul 2025,26 May 2026,ff61431f447b481090839d41c637084d,TOPIC_SQL_MCQ,SUB_TOPIC_SQL_AGGREGATIONS,Table: Discounts\r\n\r\n| discountId | product...,NaN,"[{""option_content"":""| minimum\\_discount | max...",NaN,NaN,POOL_1,DIFFICULTY_EASY,SOURCE_GPT,IN_OFFLINE_EXAM,NaN,COURSE_Introduction_to_Databases,MODULE_Aggregations_and_Group_By,UNIT_Aggregations,NaN,"CATEGORY_CODE_ANALYSIS, IN_LP, IS_PUBLIC, NIAT..."


In [6]:
len(df)

1447

In [8]:
import json

def clean_options(text):
    if not isinstance(text, str):
        return ""
    options_dictionary=json.loads(text)
    options=[item["option_content"] for item in options_dictionary]
    return "\nOptions:\n"+"\n".join(options)

df["Clean Options Data"]=df["Options Data"].apply(clean_options)
df["Question and Options"]=df["Question Content"]+df["Clean Options Data"]

In [9]:
df["Question and Options"].head()

0    Which SQL statement correctly creates a `train...
1    Which SQL statement correctly creates a `train...
2    Table: project_budgets\r\n\r\n| projectId | pr...
3    What is the correct SQL data type to store a d...
4    Table: Discounts\r\n\r\n| discountId | product...
Name: Question and Options, dtype: str

In [23]:
import chromadb

client=chromadb.Client()
client=chromadb.PersistentClient(path="./anti-examples.db")
collection=client.get_or_create_collection(name="sql_questions_already_in_db")

ids=[str(i) for i in range(len(df))]
documents=df["Question and Options"].astype(str).tolist()
metadatas=df[["Unit Tag of Question"]].to_dict(orient="records")


collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)


In [24]:
keywords="Aggregate functions"

In [25]:
results=collection.query(
    query_texts=[keywords]
)

print(results["documents"])

[['Which aggregate function is not commonly used with the GROUP BY clause?\nOptions:\nAVG()\nSUM()\nCOUNT()\nLENGTH()', 'Which aggregate function can be used with GROUP BY to find the average score for each player?\nOptions:\nSUM\nCOUNT\nAVG\nMIN', 'Which aggregate function can be used with GROUP BY to find the average score for each player?\nOptions:\nSUM\nCOUNT\nAVG\nMIN', 'What will happen if you use GROUP BY without any aggregate functions in SQL?\nOptions:\nSQL will return distinct rows based on the grouped columns\nSQL will produce a syntax error\nSQL will sum all columns by default\nSQL will return a Cartesian product', 'Which SQL aggregation function would you use to add up all values of a column?\nOptions:\nSUM\nCOUNT\nMAX\nAVG', 'What kind of function is CAST() in SQL?\nOptions:\nAggregate function\nScalar function\nString function\nSystem function', 'Which of the following statements about subqueries is true?\nOptions:\nSubqueries can only return a single value\nSubqueries m

In [26]:
# ChromaDB returns a list of lists, so we look inside the first match index [0]
for index, document in enumerate(results["documents"][0]):
    print(f"=== Matching Result #{index + 1} ===")
    print(document)
    print("\n" + "=" * 40 + "\n")

=== Matching Result #1 ===
Which aggregate function is not commonly used with the GROUP BY clause?
Options:
AVG()
SUM()
COUNT()
LENGTH()


=== Matching Result #2 ===
Which aggregate function can be used with GROUP BY to find the average score for each player?
Options:
SUM
COUNT
AVG
MIN


=== Matching Result #3 ===
Which aggregate function can be used with GROUP BY to find the average score for each player?
Options:
SUM
COUNT
AVG
MIN


=== Matching Result #4 ===
What will happen if you use GROUP BY without any aggregate functions in SQL?
Options:
SQL will return distinct rows based on the grouped columns
SQL will produce a syntax error
SQL will sum all columns by default
SQL will return a Cartesian product


=== Matching Result #5 ===
Which SQL aggregation function would you use to add up all values of a column?
Options:
SUM
COUNT
MAX
AVG


=== Matching Result #6 ===
What kind of function is CAST() in SQL?
Options:
Aggregate function
Scalar function
String function
System function


===